In [1]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [2]:
# Read the df
df = pd.read_csv('../data/train.csv')
print(f"df shape: {df.shape}")
df.sample(5)

df shape: (891216, 16)


C:\Users\Akhil\AppData\Local\Temp\ipykernel_28348\777548732.py:2: DtypeWarning: Columns (0: PRN, 1: Carrier_Doppler_hz, 2: Pseudorange_m, 3: RX_time, 4: TOW, 5: Carrier_phase, 6: EC, 7: LC, 8: PC, 9: PIP, 10: PQP, 11: TCD, 12: CN0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/train.csv')


,time,channel,PRN,Carrier_Doppler_hz,Pseudorange_m,RX_time,TOW,Carrier_phase,EC,LC,PC,PIP,PQP,TCD,CN0,spoofed
866980,108372,ch4,0,0.0,0.0,263094.22,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
200814,25101,ch6,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
690830,86353,ch6,0,0.0,0.0,175230.72,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
428764,53595,ch4,27,4491.141722,26534222.361975,174189.86,174189.771491,-392549.114912,124861.664062,132449.09375,145019.59375,-144977.875,-3478.19043,4447.57373,44.99678,1
207733,25966,ch5,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [3]:
df.shape

(891216, 16)

In [4]:
df.columns

Index(['time', 'channel', 'PRN', 'Carrier_Doppler_hz', 'Pseudorange_m',
       'RX_time', 'TOW', 'Carrier_phase', 'EC', 'LC', 'PC', 'PIP', 'PQP',
       'TCD', 'CN0', 'spoofed'],
      dtype='str')

In [5]:
df.isnull().sum()
# No missing data

time                  0
channel               0
PRN                   0
Carrier_Doppler_hz    0
Pseudorange_m         0
RX_time               0
TOW                   0
Carrier_phase         0
EC                    0
LC                    0
PC                    0
PIP                   0
PQP                   0
TCD                   0
CN0                   0
spoofed               0
dtype: int64

In [6]:
print("Start Time",df["time"].min())
print("End Time",df["time"].max())

Start Time 0
End Time 111401


Imbalanced classes 

In [7]:
df['spoofed'].value_counts()

spoofed
0    763888
1    127328
Name: count, dtype: int64

In [8]:
# First 8 rows are initializatiion rows and contain no information 
df['PRN'][8:].unique()

array(['0', '6', '2', '12', '19', 6, 0, 12, 19, 25, 11, 2, 17, 3, 24, 26,
       16, 9, 31, 4, 27, 8, 7, 20], dtype=object)

As some channels have datatype string instead of int they will be converted into integer and also drop the 

In [10]:
# Step 1 — Drop initialization rows
df = df[~df['PRN'].astype(str).str.startswith('ch')].reset_index(drop=True)

# Step 2 — Convert ALL columns to numeric
cols_to_convert = ['time', 'PRN', 'Carrier_Doppler_hz', 'Pseudorange_m', 
                   'RX_time', 'TOW', 'Carrier_phase', 
                   'EC', 'LC', 'PC', 'PIP', 'PQP', 'TCD', 'CN0']

df[cols_to_convert] = df[cols_to_convert].apply(pd.to_numeric, errors='coerce')

# Step 3 — PRN and channel should be integers specifically
df['PRN'] = df['PRN'].astype(int)

In [ ]:
df['PRN'][8:].unique()

array([ 0,  6,  2, 12, 19, 25, 11, 17,  3, 24, 26, 16,  9, 31,  4, 27,  8,
        7, 20])

In [12]:
df['channel'].unique()

<StringArray>
['ch0', 'ch1', 'ch2', 'ch3', 'ch4', 'ch5', 'ch6', 'ch7']
Length: 8, dtype: str

Early-Late Correlator Differnce(ELCD)=(EC − LC) / PC

In [13]:
df['ELCD']=(df['EC']-df['LC'])/df['PC']

In [ ]:
interval_count = 8
interval_len = len(df) // interval_count
zoom_len = min(1000, interval_len)
fig, axs = plt.subplots(2, 4, figsize=(16, 8), sharex=True)
axs = axs.flatten()

for i in range(interval_count):
    start = i * interval_len
    end = start + zoom_len
    axs[i].plot(df['ELCD'].iloc[start:end].reset_index(drop=True))
    axs[i].set_title(f'ELCD zoomed interval {i + 1} ({start}:{end})')
    axs[i].set_ylabel('ELCD')

for ax in axs[4:]:
    ax.set_xlabel('zoomed row index')

plt.tight_layout()

ValueError: Number of rows must be a positive integer, not 4.0

<Figure size 1200x2000 with 0 Axes>